In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax as dfx
import optax
import numpy as np
from jax import random
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH = 'spirals_25.npz'  # Change to spirals_75.npz, spirals_50.npz, spirals_25.npz
TRAIN_SAMPLES = None  # Use all training samples
USE_VALIDATION = False
VALIDATION_SPLIT = 0.0  # Use provided validation split

INPUT_DIM = 2        # [x, y] (time is in the data)
HIDDEN_DIM = 64
OUTPUT_DIM = 1

NUM_EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

OUTPUT_FILE = 'alpha_predictions.npy'

# ============================================================================
# DATA LOADING
# ============================================================================

def load_data(filepath, n_train=None):
    """Load spiral data from the assignment format."""
    print("Loading data...")
    data = np.load(filepath)
    
    # Assignment format: data_train, data_validation, data_test
    # Shape: (N, seq_len, 3) where 3 = [t, x, y]
    data_train = data['data_train'][:n_train] if n_train else data['data_train']
    data_val = data['data_validation']
    data_test = data['data_test']
    alpha_train = data['alpha_train'][:n_train] if n_train else data['alpha_train']
    alpha_val = data['alpha_validation']
    
    # Convert to JAX arrays and reorder to [x, y, t]
    def reorder_channels(arr):
        # Input: (batch, seq, 3) where channels are [t, x, y]
        # Output: (batch, seq, 3) where channels are [x, y, t]
        return jnp.concatenate([arr[:, :, 1:3], arr[:, :, 0:1]], axis=-1)
    
    train_data = reorder_channels(jnp.array(data_train))
    val_data = reorder_channels(jnp.array(data_val))
    test_data = reorder_channels(jnp.array(data_test))
    
    # Normalize x, y coordinates (not time)
    print("Normalizing x, y coordinates...")
    xy_mean = train_data[:, :, :2].mean(axis=(0, 1))
    xy_std = train_data[:, :, :2].std(axis=(0, 1))
    
    safe_std = xy_std + 1e-8
    train_data = train_data.at[:, :, :2].set((train_data[:, :, :2] - xy_mean) / safe_std)
    val_data = val_data.at[:, :, :2].set((val_data[:, :, :2] - xy_mean) / safe_std)
    test_data = test_data.at[:, :, :2].set((test_data[:, :, :2] - xy_mean) / safe_std)
    
    alpha_train = jnp.array(alpha_train).squeeze()
    alpha_val = jnp.array(alpha_val).squeeze()
    
    print(f"Train shape: {train_data.shape}, Alpha shape: {alpha_train.shape}")
    print(f"Val shape: {val_data.shape}, Val alpha shape: {alpha_val.shape}")
    print(f"Test shape: {test_data.shape}")
    print(f"Alpha range: [{alpha_train.min():.4f}, {alpha_train.max():.4f}]")
    
    return train_data, alpha_train, val_data, alpha_val, test_data

# ============================================================================
# GRU-ODE MODEL (OPTIMIZED WITH SCAN)
# ============================================================================

class ODEFunc(eqx.Module):
    mlp: eqx.nn.MLP
    hidden_size: int

    def __init__(self, hidden_size: int, *, key):
        self.hidden_size = hidden_size
        self.mlp = eqx.nn.MLP(
            in_size=hidden_size,
            out_size=hidden_size,
            width_size=hidden_size,
            depth=2,
            activation=jax.nn.softplus,
            key=key,
        )

    def __call__(self, t, h, args):
        return self.mlp(h)


class GRUCell(eqx.Module):
    Wz: jnp.ndarray
    Wr: jnp.ndarray
    Wh: jnp.ndarray
    
    def __init__(self, input_size, hidden_size, key):
        key_z, key_r, key_h = random.split(key, 3)
        scale = 1.0 / jnp.sqrt(hidden_size)
        self.Wz = random.normal(key_z, (hidden_size + input_size, hidden_size)) * scale
        self.Wr = random.normal(key_r, (hidden_size + input_size, hidden_size)) * scale
        self.Wh = random.normal(key_h, (hidden_size + input_size, hidden_size)) * scale
    
    def __call__(self, x, h_prev):
        combined = jnp.concatenate([h_prev, x], axis=-1)
        z = jax.nn.sigmoid(combined @ self.Wz)
        r = jax.nn.sigmoid(combined @ self.Wr)
        combined_reset = jnp.concatenate([r * h_prev, x], axis=-1)
        h_prime = jnp.tanh(combined_reset @ self.Wh)
        h = (1 - z) * h_prime + z * h_prev
        return h


class GRUODERegressor(eqx.Module):
    ode_func: ODEFunc
    gru_cell: GRUCell
    regressor: eqx.nn.Linear
    hidden_size: int

    def __init__(self, input_size: int, hidden_size: int, output_size: int, *, key):
        key_ode, key_gru, key_reg = random.split(key, 3)
        self.hidden_size = hidden_size
        self.ode_func = ODEFunc(hidden_size, key=key_ode)
        self.gru_cell = GRUCell(input_size, hidden_size, key=key_gru)
        self.regressor = eqx.nn.Linear(hidden_size, output_size, key=key_reg)

    def __call__(self, trajectory):
        """
        trajectory: (seq_len, 3) [x, y, time]
        returns: scalar prediction for alpha
        """
        h0 = jnp.zeros((self.hidden_size,), dtype=jnp.float32)
        t0 = trajectory[0, 2]
        
        solver = dfx.Dopri5()
        term = dfx.ODETerm(self.ode_func)
        
        # OPTIMIZATION: Use jax.lax.scan instead of Python loop
        def scan_step(carry, inputs):
            h_prev, t_prev = carry
            obs_curr = inputs[:2]  # x, y
            t_curr = inputs[2]     # time
            
            # Evolve via ODE
            dt = t_curr - t_prev
            target_dt0 = jnp.where(dt > 0, dt / 5.0, 0.01)
            
            solution = dfx.diffeqsolve(
                term, solver,
                t0=t_prev, t1=t_curr,
                dt0=target_dt0,
                y0=h_prev,
                max_steps=16,
            )
            
            h_ode = solution.ys[0]
            
            # Update with observation
            h_next = self.gru_cell(obs_curr, h_ode)
            
            return (h_next, t_curr), None
        
        (h_final, _), _ = jax.lax.scan(scan_step, (h0, t0), trajectory)
        
        return self.regressor(h_final).squeeze()

# ============================================================================
# TRAINING (OPTIMIZED WITH VMAP)
# ============================================================================

def loss_fn(model, batch_trajectories, batch_alphas):
    """Batched MSE loss using vmap."""
    pred_fn = jax.vmap(model)
    preds = pred_fn(batch_trajectories)
    return jnp.mean((preds - batch_alphas) ** 2)

@eqx.filter_jit
def train_step(model, opt_state, batch_trajectories, batch_alphas, optimizer):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, batch_trajectories, batch_alphas)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

@eqx.filter_jit
def evaluate_batch(model, data, alphas):
    """Efficient batched evaluation."""
    pred_fn = jax.vmap(model)
    preds = pred_fn(data)
    mae = jnp.mean(jnp.abs(preds - alphas))
    rmse = jnp.sqrt(jnp.mean((preds - alphas) ** 2))
    return mae, rmse, preds

def train_model(model, train_data, train_alphas, val_data, val_alphas, 
                num_epochs, batch_size, key, use_validation=True):
    
    optimizer = optax.adam(LEARNING_RATE)
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))
    
    num_samples = train_data.shape[0]
    steps_per_epoch = num_samples // batch_size
    train_losses = []
    val_maes = []
    
    print(f"Starting training on device: {jax.devices()[0]}")
    print(f"Steps per epoch: {steps_per_epoch}")
    
    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        key, subkey = random.split(key)
        perm = random.permutation(subkey, num_samples)
        
        shuffled_data = train_data[perm]
        shuffled_alphas = train_alphas[perm]
        
        # Truncate to fit full batches
        end_idx = steps_per_epoch * batch_size
        shuffled_data = shuffled_data[:end_idx]
        shuffled_alphas = shuffled_alphas[:end_idx]
        
        # Reshape to (num_batches, batch_size, ...)
        batched_data = shuffled_data.reshape(steps_per_epoch, batch_size, *train_data.shape[1:])
        batched_alphas = shuffled_alphas.reshape(steps_per_epoch, batch_size)
        
        epoch_loss = 0.0
        
        for i in range(steps_per_epoch):
            model, opt_state, loss = train_step(
                model, opt_state, batched_data[i], batched_alphas[i], optimizer
            )
            epoch_loss += loss.item()
            
        avg_loss = epoch_loss / steps_per_epoch
        train_losses.append(avg_loss)
        
        if use_validation:
            val_mae, val_rmse, _ = evaluate_batch(model, val_data, val_alphas)
            val_maes.append(val_mae.item())
            tqdm.write(f"Ep {epoch+1} | Loss: {avg_loss:.4f} | Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f}")
        else:
            tqdm.write(f"Ep {epoch+1} | Loss: {avg_loss:.4f}")
            
    return model, train_losses, val_maes

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # Load data
    train_data, train_alphas, val_data, val_alphas, test_data = load_data(
        DATA_PATH, TRAIN_SAMPLES
    )
    
    print(f"\nTraining samples: {len(train_data)}")
    print(f"Validation samples: {len(val_data)}")
    print(f"Test samples: {len(test_data)}")
    
    # Initialize model
    key = random.PRNGKey(RANDOM_SEED)
    key, model_key = random.split(key)
    
    model = GRUODERegressor(
        input_size=2,
        hidden_size=HIDDEN_DIM,
        output_size=OUTPUT_DIM,
        key=model_key
    )
    
    # Train
    model, train_losses, val_maes = train_model(
        model, train_data, train_alphas, val_data, val_alphas,
        NUM_EPOCHS, BATCH_SIZE, key, USE_VALIDATION
    )
    
    print("\nTraining complete! Evaluating...")
    
    # Efficient batched prediction
    @eqx.filter_jit
    def predict_all(m, d):
        return jax.vmap(m)(d)
    
    # Training metrics
    train_mae, train_rmse, train_preds = evaluate_batch(model, train_data, train_alphas)
    print(f"\nTraining Set - MAE: {train_mae:.4f}, RMSE: {train_rmse:.4f}")
    
    # Validation metrics
    if USE_VALIDATION:
        val_mae, val_rmse, val_preds = evaluate_batch(model, val_data, val_alphas)
        print(f"Validation Set - MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}")
    
    # Test predictions
    test_predictions = predict_all(model, test_data)
    
    # Save
    test_predictions_reshaped = np.array(test_predictions).reshape(-1, 1)
    np.save(OUTPUT_FILE, test_predictions_reshaped)
    print(f"\nSaved {test_predictions_reshaped.shape} predictions to {OUTPUT_FILE}")
    
    # Visualization
    if USE_VALIDATION:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        axes[0, 0].plot(train_losses)
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('MSE Loss')
        axes[0, 0].set_title('Training Loss')
        axes[0, 0].grid(True)
        
        axes[0, 1].plot(val_maes)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('MAE')
        axes[0, 1].set_title('Validation MAE')
        axes[0, 1].grid(True)
        
        axes[1, 0].scatter(train_alphas, train_preds, alpha=0.5, s=5)
        min_val = min(train_alphas.min(), train_preds.min())
        max_val = max(train_alphas.max(), train_preds.max())
        axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--')
        axes[1, 0].set_xlabel('True Alpha')
        axes[1, 0].set_ylabel('Predicted Alpha')
        axes[1, 0].set_title(f'Train (MAE: {train_mae:.4f})')
        axes[1, 0].grid(True)
        
        axes[1, 1].scatter(val_alphas, val_preds, alpha=0.5, s=5)
        min_val = min(val_alphas.min(), val_preds.min())
        max_val = max(val_alphas.max(), val_preds.max())
        axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--')
        axes[1, 1].set_xlabel('True Alpha')
        axes[1, 1].set_ylabel('Predicted Alpha')
        axes[1, 1].set_title(f'Val (MAE: {val_mae:.4f})')
        axes[1, 1].grid(True)
        
        plt.tight_layout()
        plt.savefig('training_results.png', dpi=150)
        print("Plot saved to training_results.png")
        plt.show()

Loading data...
Normalizing x, y coordinates...
Train shape: (1000, 25, 3), Alpha shape: (1000,)
Val shape: (1000, 25, 3), Val alpha shape: (1000,)
Test shape: (1000, 25, 3)
Alpha range: [-17.6583, 12.0299]

Training samples: 1000
Validation samples: 1000
Test samples: 1000
Starting training on device: TFRT_CPU_0
Steps per epoch: 62


Epochs:  10%|█         | 1/10 [00:23<03:35, 23.97s/it]

Ep 1 | Loss: 4.9622


Epochs:  20%|██        | 2/10 [00:36<02:15, 16.95s/it]

Ep 2 | Loss: 1.4980


...

Ep 8 | Loss: 0.2456


Epochs:  90%|█████████ | 9/10 [01:47<00:10, 10.69s/it]

Ep 9 | Loss: 0.2075


Epochs: 100%|██████████| 10/10 [01:57<00:00, 11.76s/it]


Ep 10 | Loss: 0.2403

Training complete! Evaluating...

Training Set - MAE: 0.3278, RMSE: 0.5411

Saved (1000, 1) predictions to alpha_predictions.npy
